# k04 — Final models (STAGE2_DESIGN_FROZEN_v1.1 §8)
Per set: all train_01 recordings (SHA-256 checked), selected configurations from k03, 5 seeds; thresholds (max-F1, 1 and 5 FA/h) from pooled out-of-fold validation scores of k03 only. Saves checkpoints, LightGBM boosters, normalisation statistics, thresholds. **No test data is extracted or scored.**

In [ ]:
import os, glob
os.makedirs('/kaggle/working/code', exist_ok=True)
FILES = {'feats.py': 'import numpy as np, pandas as pd, re, os, time\nW = 64\nHEXV = np.full(256, 0, dtype=np.uint8)\nfor i, ch in enumerate(\'0123456789abcdef\'):\n    HEXV[ord(ch)] = i; HEXV[ord(ch.upper())] = i\nPOP = np.array([bin(i).count(\'1\') for i in range(256)], dtype=np.uint8)\n\ndef slog(x):\n    return np.sign(x) * np.log1p(np.abs(x))\n\ndef load_file(path):\n    df = pd.read_csv(path, dtype={\'arbitration_id\': str, \'data_field\': str, \'attack\': np.int8}, keep_default_na=True)\n    ts = df[\'timestamp\'].to_numpy(np.float64)\n    ids = df[\'arbitration_id\'].str.rjust(3, \'0\').str[-3:]\n    ib = np.frombuffer(\'\'.join(ids.tolist()).encode(\'ascii\'), dtype=np.uint8).reshape(-1, 3)\n    idv = HEXV[ib].astype(np.int32)\n    id_int = idv[:, 0] * 256 + idv[:, 1] * 16 + idv[:, 2]\n    d = df[\'data_field\'].fillna(\'\')\n    plen = (d.str.len().to_numpy() // 2).clip(0, 8).astype(np.int8)\n    d16 = d.str[:16].str.ljust(16, \'0\')\n    db = np.frombuffer(\'\'.join(d16.tolist()).encode(\'ascii\'), dtype=np.uint8).reshape(-1, 16)\n    nib = HEXV[db]\n    pay = (nib[:, 0::2] * 16 + nib[:, 1::2]).astype(np.uint8)\n    posmask = np.arange(8)[None, :] < plen[:, None]\n    pay = np.where(posmask, pay, 0).astype(np.uint8)\n    y = df[\'attack\'].to_numpy(np.int8)\n    return ts, id_int, plen, pay, y\n\ndef per_frame_globals(ts, id_int, plen, pay):\n    n = len(ts); idx = np.arange(n)\n    order = np.lexsort((idx, id_int))\n    prev = np.full(n, -1, dtype=np.int64)\n    same = np.r_[False, id_int[order][1:] == id_int[order][:-1]]\n    prev[order[same]] = order[np.flatnonzero(same) - 1]\n    has = prev >= 0\n    pp = np.where(has, prev, 0)\n    dt_same = np.where(has, ts - ts[pp], 0.0)\n    x = pay ^ pay[pp]\n    ham = np.where(has, POP[x].sum(1), 0).astype(np.float32)\n    maxlen = np.maximum(plen, plen[pp]).astype(np.float32)\n    chg = np.where(has, (x != 0).sum(1) / np.maximum(maxlen, 1), 0).astype(np.float32)\n    lenchg = np.where(has, plen != plen[pp], False)\n    ent = np.zeros(n, dtype=np.float32)\n    for s in range(0, n, 500000):\n        b = pay[s:s + 500000]; L = plen[s:s + 500000].astype(np.int32)\n        valid = np.arange(8)[None, :] < L[:, None]\n        eq = (b[:, :, None] == b[:, None, :]) & valid[:, :, None] & valid[:, None, :]\n        c = eq.sum(2).astype(np.float32)\n        Lf = np.maximum(L, 1).astype(np.float32)[:, None]\n        with np.errstate(divide=\'ignore\', invalid=\'ignore\'):\n            term = np.where(valid, np.log2(np.where(c > 0, c, 1) / Lf), 0.0)\n        ent[s:s + 500000] = -(term.sum(1) / Lf[:, 0])\n    return prev, dt_same, ham, chg, ent, lenchg\n\nFRAME_NAMES = [\'plen\', \'dt_prev_any\', \'dt_same\', \'no_prev_same_in_window\', \'hamming\', \'changed_frac\', \'entropy\', \'same_as_prev\']\nNODE_NAMES = [\'count\', \'first_pos\', \'last_pos\', \'ia_mean\', \'ia_min\', \'ia_max\', \'ia_missing\', \'plen_mean\', \'plen_max\', \'plen_changes\', \'ham_mean\', \'chg_mean\', \'ent_mean\']\nGLOBAL_NAMES = [\'duration\', \'distinct_ids\', \'distinct_transitions\', \'fps\']\n\ndef windows_for_file(path, stride, id_perm=None):\n    ts, id_int, plen, pay, y = load_file(path)\n    if id_perm is not None:\n        id_int = id_perm[id_int]\n    n = len(ts)\n    if n < W:\n        return None\n    prev, dt_same_g, ham_g, chg_g, ent_g, lenchg_g = per_frame_globals(ts, id_int, plen, pay)\n    starts = np.arange(0, n - W + 1, stride)\n    nw = len(starts)\n    I = starts[:, None] + np.arange(W)[None, :]\n    ok = prev[I] >= starts[:, None]\n    tsw = ts[I]\n    dtp = np.diff(tsw, axis=1, prepend=tsw[:, :1])\n    idw = id_int[I]\n    fr = np.zeros((nw, W, len(FRAME_NAMES)), dtype=np.float32)\n    fr[..., 0] = plen[I] / 8.0\n    fr[..., 1] = slog(dtp * 1000)\n    fr[..., 2] = np.where(ok, slog(dt_same_g[I] * 1000), 0)\n    fr[..., 3] = ~ok\n    fr[..., 4] = np.where(ok, ham_g[I] / 64.0, 0)\n    fr[..., 5] = np.where(ok, chg_g[I], 0)\n    fr[..., 6] = ent_g[I] / 3.0\n    fr[:, 1:, 7] = idw[:, 1:] == idw[:, :-1]\n    # nodes\n    key = (np.arange(nw)[:, None] * 4096 + idw).ravel()\n    uk, inv = np.unique(key, return_inverse=True)\n    inv = inv.reshape(nw, W)\n    win_of_node = uk // 4096\n    node_first = np.searchsorted(win_of_node, np.arange(nw))\n    local = inv - node_first[:, None]\n    nn = np.bincount(win_of_node, minlength=nw)\n    G = len(uk)\n    fl = inv.ravel()\n    pos = np.broadcast_to(np.arange(W), (nw, W)).ravel()\n    okf = ok.ravel()\n    def agg_sum(v, m=None):\n        return np.bincount(fl, weights=(v if m is None else v * m), minlength=G)\n    order = np.argsort(fl, kind=\'stable\'); fs = fl[order]\n    bnd = np.flatnonzero(np.r_[True, fs[1:] != fs[:-1]])\n    def agg_min(v): return np.minimum.reduceat(v[order], bnd)\n    def agg_max(v): return np.maximum.reduceat(v[order], bnd)\n    cnt = np.bincount(fl, minlength=G).astype(np.float32)\n    iak = np.where(okf, slog(dt_same_g[I].ravel() * 1000), np.nan)\n    nia = agg_sum(okf.astype(np.float64))\n    has_ia = nia > 0\n    ia_mean = np.where(has_ia, agg_sum(np.nan_to_num(iak)) / np.maximum(nia, 1), 0)\n    ia_min = np.where(has_ia, agg_min(np.where(okf, iak, np.inf)), 0)\n    ia_max = np.where(has_ia, agg_max(np.where(okf, iak, -np.inf)), 0)\n    pl = (plen[I].ravel()).astype(np.float64)\n    nd = np.zeros((G, len(NODE_NAMES)), dtype=np.float32)\n    nd[:, 0] = cnt / W\n    nd[:, 1] = agg_min(pos.astype(np.float64)) / W\n    nd[:, 2] = agg_max(pos.astype(np.float64)) / W\n    nd[:, 3] = ia_mean; nd[:, 4] = ia_min; nd[:, 5] = ia_max\n    nd[:, 6] = ~has_ia\n    nd[:, 7] = agg_sum(pl) / cnt / 8.0\n    nd[:, 8] = agg_max(pl) / 8.0\n    nd[:, 9] = agg_max(pl) != agg_min(pl)\n    nd[:, 10] = np.where(has_ia, agg_sum(ham_g[I].ravel().astype(np.float64), okf) / np.maximum(nia, 1) / 64.0, 0)\n    nd[:, 11] = np.where(has_ia, agg_sum(chg_g[I].ravel().astype(np.float64), okf) / np.maximum(nia, 1), 0)\n    nd[:, 12] = agg_sum(ent_g[I].ravel().astype(np.float64)) / cnt / 3.0\n    node = np.zeros((nw, W, len(NODE_NAMES)), dtype=np.float32)\n    node[win_of_node, np.arange(G) - node_first[win_of_node]] = nd\n    # edges: local src/dst per transition\n    src = local[:, :-1].astype(np.uint8); dst = local[:, 1:].astype(np.uint8)\n    tr = np.unique((np.arange(nw)[:, None] * 4096 + local[:, :-1] * 64 + local[:, 1:]).ravel())\n    ntr = np.bincount(tr // 4096, minlength=nw)\n    dur = tsw[:, -1] - tsw[:, 0]\n    glob = np.stack([slog(dur * 1000), nn / W, ntr / 63.0, slog(W / np.maximum(dur, 1e-6))], 1).astype(np.float32)\n    lab = (y[I].max(1) > 0).astype(np.int8)\n    nattack = y[I].sum(1).astype(np.int16)\n    return dict(frame=fr.astype(np.float16), node=node.astype(np.float16), nmask=(np.arange(W)[None, :] < nn[:, None]),\n                src=src, dst=dst, glob=glob, y=lab, nattack=nattack, starts=starts.astype(np.int64), t0=tsw[:, 0], t1=tsw[:, -1])\n\nSTRUCT_NAMES = [\'transition_entropy\', \'unique_transition_ratio\', \'self_loop_ratio\', \'mean_out_degree\',\n                \'max_out_degree\', \'max_in_degree\', \'degree_entropy\', \'density\']\n\ndef structural_features(src, dst, nmask):\n    """Explicit structural/topological summaries of each window\'s directed transition multigraph.\n    src, dst: (nw, 63) local node indices of consecutive frames; nmask: (nw, 64) valid nodes.\n    Uses only ID-free graph structure (local node indices are arbitrary labels)."""\n    nw, E = src.shape\n    n = nmask.sum(1).astype(np.float64)\n    s = src.astype(np.int64); d = dst.astype(np.int64)\n    w = np.repeat(np.arange(nw), E)\n    key = w * 4096 + (s * 64 + d).ravel()\n    uk, cnt = np.unique(key, return_counts=True)\n    uw = uk // 4096; us = (uk % 4096) // 64; ud = (uk % 4096) % 64\n    p = cnt / float(E)\n    ent = np.bincount(uw, weights=-p * np.log2(p), minlength=nw) / np.log2(E)\n    uniq = np.bincount(uw, minlength=nw) / float(E)\n    selfr = (s == d).sum(1) / float(E)\n    ns = us != ud\n    outdeg = np.bincount(uw[ns] * 64 + us[ns], minlength=nw * 64).reshape(nw, 64).astype(np.float64)\n    indeg = np.bincount(uw[ns] * 64 + ud[ns], minlength=nw * 64).reshape(nw, 64).astype(np.float64)\n    e_ns = np.bincount(uw[ns], minlength=nw).astype(np.float64)\n    mean_out = np.where(n > 0, e_ns / np.maximum(n, 1), 0)\n    tot = outdeg + indeg; ts = tot.sum(1, keepdims=True)\n    pd_ = np.where(ts > 0, tot / np.maximum(ts, 1), 0)\n    with np.errstate(divide=\'ignore\', invalid=\'ignore\'):\n        h = -(np.where(pd_ > 0, pd_ * np.log2(np.where(pd_ > 0, pd_, 1)), 0)).sum(1)\n    deg_ent = np.where(n > 1, h / np.log2(np.maximum(n, 2)), 0)\n    dens = np.where(n > 1, e_ns / np.maximum(n * (n - 1), 1), 0)\n    return np.stack([ent, uniq, selfr, mean_out, outdeg.max(1), indeg.max(1), deg_ent, dens], 1).astype(np.float32)\n', 'models.py': "import torch, torch.nn as nn, numpy as np, time\nW = 64\n\ndef mlp(i, h, o, drop):\n    return nn.Sequential(nn.Linear(i, h), nn.ReLU(), nn.Dropout(drop), nn.Linear(h, o))\n\nclass Head(nn.Module):\n    def __init__(self, h, g, drop):\n        super().__init__(); self.rho = mlp(3 * h + g, h, 1, drop)\n    def forward(self, H, mask, glob):\n        m = mask.unsqueeze(-1).float()\n        s = (H * m).sum(1); mean = s / m.sum(1).clamp(min=1)\n        mx = H.masked_fill(m == 0, -1e4).max(1).values\n        return self.rho(torch.cat([s, mean, mx, glob], 1)).squeeze(-1)\n\nclass DeepSets(nn.Module):\n    def __init__(self, f, h, g, drop=0.0):\n        super().__init__()\n        self.phi = nn.Sequential(nn.Linear(f, h), nn.ReLU(), nn.Dropout(drop), nn.Linear(h, h), nn.ReLU())\n        self.head = Head(h, g, drop)\n    def forward(self, b):\n        return self.head(self.phi(b['node']), b['nmask'], b['glob'])\n\nclass SAGELayer(nn.Module):\n    def __init__(self, i, o):\n        super().__init__(); self.self_lin = nn.Linear(i, o); self.nei_lin = nn.Linear(i, o, bias=False)\n    def forward(self, H, A):\n        # A[b, src, dst] = weight; aggregate incoming neighbours of each dst node (weighted mean)\n        agg = torch.bmm(A.transpose(1, 2), H)\n        deg = A.sum(1).unsqueeze(-1)\n        agg = agg / deg.clamp(min=1e-9)\n        return self.self_lin(H) + self.nei_lin(agg)\n\nclass GraphSAGE(nn.Module):\n    def __init__(self, f, h, g, drop=0.0):\n        super().__init__()\n        self.l1 = SAGELayer(f, h); self.l2 = SAGELayer(h, h); self.drop = nn.Dropout(drop)\n        self.head = Head(h, g, drop)\n    def forward(self, b):\n        A = b['adj']; m = b['nmask'].unsqueeze(-1).float()\n        H = torch.relu(self.l1(b['node'], A)) * m\n        H = torch.relu(self.l2(self.drop(H), A)) * m\n        return self.head(H, b['nmask'], b['glob'])\n\nclass GRUNet(nn.Module):\n    def __init__(self, f, h, g, drop=0.0):\n        super().__init__()\n        self.gru = nn.GRU(f, h, batch_first=True); self.out = mlp(2 * h + g, h, 1, drop)\n    def forward(self, b):\n        O, hn = self.gru(b['frame'])\n        return self.out(torch.cat([hn[-1], O.mean(1), b['glob']], 1)).squeeze(-1)\n\ndef nparams(m):\n    return sum(p.numel() for p in m.parameters())\n\ndef build_adj(src, dst, B, device):\n    A = torch.zeros(B, W, W, device=device)\n    bi = torch.arange(B, device=device).unsqueeze(1).expand_as(src)\n    A.index_put_((bi.reshape(-1), src.reshape(-1).long(), dst.reshape(-1).long()), torch.full((src.numel(),), 1.0 / 63, device=device), accumulate=True)\n    return A\n\ndef rewire_dst(dst, gen):\n    # degree-preserving directed rewiring: permute destination endpoints among a window's 63 edges\n    # (every source keeps its out-degree, every destination keeps its in-degree, multiplicities included)\n    r = torch.rand(dst.shape, generator=gen, device=dst.device)\n    perm = r.argsort(1)\n    return torch.gather(dst, 1, perm)\n\ndef edge_change_fraction(src, dst, dst2):\n    # fraction of the 63 directed edges (as a multiset per window) not present in the original\n    B = src.shape[0]\n    k1 = (src.long() * 64 + dst.long()).sort(1).values\n    k2 = (src.long() * 64 + dst2.long()).sort(1).values\n    fr = []\n    for i in range(B):\n        a, ca = torch.unique(k1[i], return_counts=True); b2, cb = torch.unique(k2[i], return_counts=True)\n        common = 0\n        d = dict(zip(a.tolist(), ca.tolist()))\n        for kk, cc in zip(b2.tolist(), cb.tolist()):\n            common += min(cc, d.get(kk, 0))\n        fr.append(1 - common / 63.0)\n    return float(np.mean(fr))\n", 'exp_final.py': '"""Stage 4 final models (STAGE2_DESIGN_FROZEN_v1.1 §8). Reads ONLY train_01 + k03 tuning outputs. No test data.\nUsage: python exp_final.py --sets set_01,set_03 --device cuda:0"""\nimport os, sys, re, json, time, glob, hashlib, argparse, traceback, zipfile, math\nimport numpy as np, torch\nfrom sklearn.metrics import precision_recall_curve\nsys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))\nimport feats, models\n\nap = argparse.ArgumentParser()\nap.add_argument(\'--sets\', required=True); ap.add_argument(\'--device\', default=\'cuda:0\')\nap.add_argument(\'--out\', default=\'/kaggle/working/final\'); ap.add_argument(\'--lgb_threads\', type=int, default=2)\nap.add_argument(\'--seeds\', default=\'0,1,2,3,4\')\nargs = ap.parse_args()\nDEV = args.device; OUT = args.out; os.makedirs(OUT, exist_ok=True)\nSETS = args.sets.split(\',\'); SEEDS = [int(s) for s in args.seeds.split(\',\')]; BS = 1024\nZIP = glob.glob(\'/kaggle/input/**/can-train-and-test-v1.zip\', recursive=True)[0]\nHASHES = glob.glob(\'/kaggle/input/**/file_hashes_sha256.csv\', recursive=True)[0]\nDATA = \'/kaggle/temp/data\'\nLOG = open(os.path.join(OUT, f\'log_{"_".join(SETS)}.txt\'), \'a\')\ndef log(*a):\n    s = time.strftime(\'%H:%M:%S \') + \' \'.join(str(x) for x in a); print(s, flush=True); LOG.write(s + \'\\n\'); LOG.flush()\n\ndef extract_train(st):\n    H = {}\n    for line in open(HASHES).read().strip().split(\'\\n\')[1:]:\n        rel, size, sha = line.split(\',\'); H[rel] = (int(size), sha)\n    os.makedirs(DATA, exist_ok=True)\n    with zipfile.ZipFile(ZIP) as z:\n        names = [n for n in z.namelist() if n.startswith(f\'can-train-and-test/{st}/train_01/\') and n.endswith(\'.csv\')]\n        assert names and all(\'/train_01/\' in n for n in names)\n        for n in names:\n            if not os.path.exists(os.path.join(DATA, n)): z.extract(n, DATA)\n    files = sorted(glob.glob(os.path.join(DATA, \'can-train-and-test\', st, \'train_01\', \'*.csv\')))\n    for p in files:\n        rel = p.split(\'can-train-and-test/\')[1]; hh = hashlib.sha256()\n        with open(p, \'rb\') as f:\n            for b in iter(lambda: f.read(8 << 20), b\'\'): hh.update(b)\n        assert (os.path.getsize(p), hh.hexdigest()) == H[rel], \'hash mismatch \' + rel\n    return files\n\nNF, NN, NG = len(feats.FRAME_NAMES), len(feats.NODE_NAMES), len(feats.GLOBAL_NAMES)\ndef make(name, h, drop):\n    if name in (\'GraphSAGE\', \'GraphSAGE_rewired\'): return models.GraphSAGE(NN, h, NG, drop)\n    if name == \'DeepSets\': return models.DeepSets(NN, h, NG, drop)\n    if name == \'GRU\': return models.GRUNet(NF, h, NG, drop)\n\ndef flat_features(D, with_struct):\n    m = D[\'nmask\'][..., None]; x = D[\'node\'].astype(np.float32); cnt = m.sum(1).clip(1)\n    mean = (x * m).sum(1) / cnt; std = np.sqrt(((x - mean[:, None]) ** 2 * m).sum(1) / cnt)\n    mn = np.where(m, x, np.inf).min(1); mx = np.where(m, x, -np.inf).max(1)\n    X = [mean, std, mn, mx, D[\'glob\']]\n    if with_struct: X.append(D[\'struct\'])\n    return np.concatenate(X, 1).astype(np.float32)\n\ndef norm_stats_np(D):\n    f = D[\'frame\'].astype(np.float32).reshape(-1, NF); msk = D[\'nmask\'].reshape(-1)\n    n = D[\'node\'].astype(np.float32).reshape(-1, NN)[msk]\n    return {\'fm\': f.mean(0), \'fs\': f.std(0) + 1e-6, \'nm\': n.mean(0), \'ns\': n.std(0) + 1e-6, \'gm\': D[\'glob\'].mean(0), \'gs\': D[\'glob\'].std(0) + 1e-6}\n\ndef to_gpu(D, N):\n    T = lambda a: torch.tensor(a, dtype=torch.float32, device=DEV)\n    fm, fs, nm_, ns, gm, gs = T(N[\'fm\']), T(N[\'fs\']), T(N[\'nm\']), T(N[\'ns\']), T(N[\'gm\']), T(N[\'gs\'])\n    out = {}\n    out[\'frame\'] = ((torch.from_numpy(D[\'frame\'].astype(np.float32)).to(DEV) - fm) / fs).half()\n    msk = torch.from_numpy(D[\'nmask\']).to(DEV)\n    out[\'node\'] = (((torch.from_numpy(D[\'node\'].astype(np.float32)).to(DEV) - nm_) / ns) * msk.unsqueeze(-1)).half()\n    out[\'nmask\'] = msk; out[\'glob\'] = (torch.from_numpy(D[\'glob\']).to(DEV) - gm) / gs\n    out[\'src\'] = torch.from_numpy(D[\'src\']).to(DEV); out[\'dst\'] = torch.from_numpy(D[\'dst\']).to(DEV)\n    out[\'y\'] = torch.from_numpy(D[\'y\'].astype(np.float32)).to(DEV)\n    return out\n\ndef batch(T, ix, rewire, gen):\n    b = {\'frame\': T[\'frame\'][ix].float(), \'node\': T[\'node\'][ix].float(), \'nmask\': T[\'nmask\'][ix], \'glob\': T[\'glob\'][ix]}\n    dst = T[\'dst\'][ix]\n    if rewire: dst = models.rewire_dst(dst, gen)\n    b[\'adj\'] = models.build_adj(T[\'src\'][ix], dst, len(ix), DEV)\n    return b\n\ndef thresholds_from_oof(scores, y, val_hours):\n    neg = np.sort(scores[y == 0])[::-1]; pos = scores[y == 1]\n    out = {\'val_hours\': val_hours, \'n_val_windows\': int(len(y)), \'n_val_pos\': int(y.sum())}\n    p, r, t = precision_recall_curve(y, scores)\n    f1 = 2 * p[:-1] * r[:-1] / np.maximum(p[:-1] + r[:-1], 1e-12)\n    i = int(np.argmax(f1))\n    out[\'f1\'] = {\'threshold\': float(t[i]), \'op\': \'>=\', \'oof_f1\': float(f1[i]), \'oof_precision\': float(p[i]), \'oof_recall\': float(r[i])}\n    for rate in (1, 5):\n        k = int(math.floor(rate * val_hours))\n        thr = float(neg[k]) if k < len(neg) else float(neg[-1]) - 1e-6\n        fp = int((neg > thr).sum()); rec = float((pos > thr).mean()) if len(pos) else float(\'nan\')\n        out[f\'fa{rate}\'] = {\'threshold\': thr, \'op\': \'>\', \'allowed_fp\': k, \'oof_fp\': fp, \'oof_recall\': rec}\n    return out\n\ndef run_set(st):\n    sd = os.path.join(OUT, st); os.makedirs(sd, exist_ok=True)\n    if os.path.exists(os.path.join(sd, \'final_meta.json\')):\n        log(st, \'already done\'); return\n    tsel = glob.glob(f\'/kaggle/input/**/tune/{st}/selection.json\', recursive=True)[0]\n    SEL = json.load(open(tsel)); OOF = np.load(os.path.join(os.path.dirname(tsel), \'oof_scores.npz\'))\n    t0 = time.time(); files = extract_train(st); log(st, \'train files\', len(files), \'verify s\', round(time.time() - t0, 1))\n    t0 = time.time(); F = {}; hours = 0.0\n    for p in files:\n        d = feats.windows_for_file(p, 32); d[\'struct\'] = feats.structural_features(d[\'src\'], d[\'dst\'], d[\'nmask\']); F[p] = d\n        hours += float(d[\'t1\'].max() - d[\'t0\'].min()) / 3600.0\n    keys = [\'frame\', \'node\', \'nmask\', \'src\', \'dst\', \'glob\', \'y\', \'struct\']\n    D = {k: np.concatenate([F[p][k] for p in files]) for k in keys}\n    log(st, \'features s\', round(time.time() - t0, 1), \'windows\', len(D[\'y\']), \'pos\', int(D[\'y\'].sum()), \'train hours\', round(hours, 3))\n    # thresholds from pooled out-of-fold validation scores (tuning, seed 0)\n    y_oof = np.concatenate([OOF[\'y_f1\'], OOF[\'y_f2\']]).astype(np.int8)\n    TH = {}\n    for name in (\'Rule\', \'LightGBM\', \'LightGBM_S\', \'DeepSets\', \'GRU\', \'GraphSAGE\', \'GraphSAGE_rewired\'):\n        s = np.concatenate([OOF[f\'{name}_f1\'], OOF[f\'{name}_f2\']]).astype(np.float64)\n        TH[name] = thresholds_from_oof(s, y_oof, hours)\n        log(st, \'thresholds\', name, {k: TH[name][k] for k in (\'f1\', \'fa1\', \'fa5\')})\n    json.dump(TH, open(os.path.join(sd, \'thresholds.json\'), \'w\'), indent=1)\n    N = norm_stats_np(D); np.savez(os.path.join(sd, \'norm_stats.npz\'), **N)\n    META = {\'set\': st, \'train_files\': [os.path.basename(p) for p in files], \'train_windows\': int(len(D[\'y\'])), \'train_pos\': int(D[\'y\'].sum()),\n            \'train_hours\': hours, \'seeds\': SEEDS, \'models\': {}}\n    META[\'models\'][\'Rule\'] = {\'feature_index\': SEL[\'Rule\'][\'feature_index\'], \'sign\': SEL[\'Rule\'][\'sign\'], \'layout\': \'56-feature LightGBM layout\'}\n    # LightGBM / LightGBM+S\n    import lightgbm as lgb\n    for name, ws in ((\'LightGBM\', False), (\'LightGBM_S\', True)):\n        X = flat_features(D, ws); y = D[\'y\']; pos = y.sum(); neg = len(y) - pos\n        rec = {\'config\': SEL[name][\'config\'], \'n_estimators\': SEL[name][\'final_epochs\'], \'n_features\': int(X.shape[1]), \'seeds\': {}}\n        for seed in SEEDS:\n            t = time.time()\n            clf = lgb.LGBMClassifier(n_estimators=SEL[name][\'final_epochs\'], scale_pos_weight=neg / max(pos, 1), n_jobs=args.lgb_threads, verbose=-1, random_state=seed, **SEL[name][\'config\'])\n            clf.fit(X, y)\n            clf.booster_.save_model(os.path.join(sd, f\'{name}_seed{seed}.txt\'))\n            rec[\'seeds\'][seed] = {\'fit_s\': round(time.time() - t, 1)}\n        META[\'models\'][name] = rec; log(st, name, \'saved\', rec[\'seeds\'])\n    # neural\n    T = to_gpu(D, N); n = len(D[\'y\'])\n    pos = float(T[\'y\'].sum()); neg = n - pos\n    for name in (\'DeepSets\', \'GRU\', \'GraphSAGE\', \'GraphSAGE_rewired\'):\n        sel = SEL[name]; cfg = sel[\'config\']; h = sel[\'hidden\']; E = sel[\'final_epochs\']\n        rec = {\'config\': cfg, \'hidden\': h, \'epochs\': E, \'seeds\': {}}\n        for seed in SEEDS:\n            try:\n                torch.manual_seed(seed); np.random.seed(seed)\n                gen = torch.Generator(device=DEV); gen.manual_seed(seed)\n                m = make(name, h, cfg[\'dropout\']).to(DEV)\n                opt = torch.optim.AdamW(m.parameters(), lr=cfg[\'lr\'])\n                lossf = torch.nn.BCEWithLogitsLoss(pos_weight=torch.tensor(neg / max(pos, 1.0), device=DEV))\n                rew = name == \'GraphSAGE_rewired\'; losses = []; t = time.time()\n                for ep in range(E):\n                    m.train(); perm = torch.randperm(n, device=DEV, generator=gen); tot = 0.0\n                    for s in range(0, n, BS):\n                        ix = perm[s:s + BS]\n                        loss = lossf(m(batch(T, ix, rew, gen)), T[\'y\'][ix]); opt.zero_grad(); loss.backward(); opt.step(); tot += loss.item() * len(ix)\n                    losses.append(round(tot / n, 6))\n                torch.save(m.state_dict(), os.path.join(sd, f\'{name}_seed{seed}.pt\'))\n                rec[\'seeds\'][seed] = {\'train_loss\': losses, \'fit_s\': round(time.time() - t, 1), \'params\': models.nparams(m)}\n            except Exception as e:\n                rec[\'seeds\'][seed] = {\'error\': repr(e)}; log(st, name, seed, \'ERROR\', repr(e), traceback.format_exc()[-800:])\n            log(st, name, \'seed\', seed, {k: v for k, v in rec[\'seeds\'][seed].items() if k != \'train_loss\'}, \'last loss\', rec[\'seeds\'][seed].get(\'train_loss\', [None])[-1])\n        META[\'models\'][name] = rec\n    json.dump(META, open(os.path.join(sd, \'final_meta.json\'), \'w\'), indent=1, default=str)\n    log(st, \'DONE\')\n\nfor st in SETS:\n    try:\n        run_set(st)\n    except Exception as e:\n        log(st, \'FATAL\', repr(e), traceback.format_exc()[-2000:])\n'}
for n, s in FILES.items():
    open('/kaggle/working/code/' + n, 'w').write(s)
os.makedirs('/kaggle/temp', exist_ok=True)
print(sorted(os.listdir('/kaggle/working/code')))
print(glob.glob('/kaggle/input/**/can-train-and-test-v1.zip', recursive=True))
print(sorted(glob.glob('/kaggle/input/**/tune/set_0*/selection.json', recursive=True)))
assert len(glob.glob('/kaggle/input/**/tune/set_0*/selection.json', recursive=True)) == 4

In [ ]:
import subprocess, sys, time
cmd = lambda sets, dev: [sys.executable, '/kaggle/working/code/exp_final.py', '--sets', sets, '--device', dev, '--lgb_threads', '2']
t0 = time.time()
pA = subprocess.Popen(cmd('set_01,set_03', 'cuda:0'))
pB = subprocess.Popen(cmd('set_02,set_04', 'cuda:1'))
print('exit codes', pA.wait(), pB.wait(), 'hours', round((time.time() - t0) / 3600, 2))

In [ ]:
import json, glob, os
for p in sorted(glob.glob('/kaggle/working/final/set_0*/final_meta.json')):
    M = json.load(open(p)); print(p, 'windows', M['train_windows'], 'hours', round(M['train_hours'], 3))
    for k, v in M['models'].items():
        errs = [s for s, r in v.get('seeds', {}).items() if 'error' in r]
        print('  ', k, 'errors', errs, {kk: v[kk] for kk in v if kk in ('hidden', 'epochs', 'n_estimators', 'feature_index', 'sign')})
for p in sorted(glob.glob('/kaggle/working/final/set_0*/thresholds.json')):
    print(p, json.dumps({k: {kk: round(vv['threshold'], 5) for kk, vv in v.items() if isinstance(vv, dict)} for k, v in json.load(open(p)).items()}))
print(len(glob.glob('/kaggle/working/final/set_0*/*.pt')), 'neural checkpoints;', len(glob.glob('/kaggle/working/final/set_0*/*.txt')), 'lightgbm models')